Урок в прозе: https://proproprogs.ru/python_oop/deskriptory-data-descriptor-i-non-data-descriptor

Телеграм-канал: https://t.me/python_selfedu

Введение в декораторы функций: https://youtu.be/v0qZPplzwUQ

In [1]:
class Point3D:
    def __init__(self, x, y, z):
        self.x = x # тут мы вызываем наши объекты-свойства, так удобнее, так как сразу срабатывает проверка
        self.y = y
        self.z = z

    @classmethod
    def verify_coord(cls, coord):
        if type(coord) != int:
            raise TypeError('ккординта олжна быть целым числом')

    @property
    def x(self):
        return self._x
    @x.setter
    def x(self, coord):
        self.verify_coord(coord)
        self._x = coord

    @property
    def y(self):
        return self._y
    @y.setter
    def y(self, coord):
        self.verify_coord(coord)
        self._y = coord

    @property
    def z(self):
        return self._z
    @z.setter
    def z(self, coord):
        self.verify_coord(coord)
        self._z = coord


p = Point3D(1,2,3)
p.__dict__

{'_x': 1, '_y': 2, '_z': 3}

В нашем классе мы трижды прописали объекты-свойства. Если бы их было больше 3, это был бы ад. Тут нам помогут дескрипторы

Дескрипторы это класс, котоый содержт специальный метод __get__ (тогда это дескриптор не данных non-data descriptor), или, если еще методы __set__ и __del__ (дескриптор данных data descriptor)

In [ ]:
class A:
    '''Дескриптор НЕ данных, умеют только считывать информацию
    если мы попытаемся через такой дескриптор переопределить атрибут, то тупо создастся атрибут с именем дескриптора
    '''
    def __get__(self, instance, owner):
        return ...

class B:
    """Дескриптор данных. приоритет обращения к дескриптору данных выше, чем к пространству __dict__.
    То есть при наличии одинаковых имен, вызовется дескриптор, а не локальное свойства"""
    def __get__(self, instance, owner):
        return ...

    def __set__(self, instance, value):
        ...

    def __del__(self):
        ...


Так как мы договорились, что координаты это целые числа, то и интерфейс взаимодействия с ними будет единым. Опишем его в классе Integer

In [6]:
class Integer:
    def __set_name__(self, owner, name):
        # срабатывает автоматически при создании экземпляра класса (мы создаем его в самом начале Point3D)
        # self - ссылка на экземпляра класса, который мы объявили внутри Point3D, то есть на дескриптор
        # owner - ссылка на класс Point3D
        # name - имя переменной, которой присваивается экземпляр класса (то есть x, y или z)
        # в итоге когда мы пишем x = Integer(), то у этого x появляется локальное свойство name со значением _x

        self.name="_"+name

    def __get__(self, instance, owner):
        # return instance.__dict__[self.name]
        return getattr(instance, self.name) # так грамотнее, чем обращение напрямую к списку __dict__

    def __set__(self, instance, value):
        # self - ссылка на сам дескриптор
        # instance - ссылка на экземпляр класса Point3D, из которого вызван дескриптор
        # value - значение, которое мы присваиваем
        self.verify_coord(value)
        print(f'__set__:{self.name}={value}')
        # instance.__dict__[self.name]=value
        setattr(instance, self.name, value) # так грамотнее, чем обращение напрямую к списку __dict__

    @classmethod
    def verify_coord(cls, coord):
        # по сравнению с предыдущим вариантом, проверку нам пришлось перенести внутрь класса дескриптора
        if type(coord) != int:
            raise TypeError('ккординта олжна быть целым числом')


class Point3D:
    # создаем дескрипторы для каждой координаты внутри класса Point3D
    x = Integer()
    y = Integer()
    z = Integer()

    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z


p = Point3D(1,2,3)
# Когда мы создаем экземпляр класса Point3D, вызывается метод __init__,
# внутри него идет обращение к дескрипторам x, y, z, которые являются экземплярами класса Integer
# В момент присваивания в деескрипторе срабатывает сеттер __set__
# в нем в словаре атрибутов по ключу self.name присваивается желаемое значение
p.__dict__

__set__:_x=1
__set__:_y=2
__set__:_z=3


{'_x': 1, '_y': 2, '_z': 3}

{'__x': 1, '__y': 2, '__radius': 3}